<a href="https://colab.research.google.com/github/sergiocostaifes/PPCOMP_DM/blob/main/notebooks/03_window_5min_base.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NB03 — Construção de Janelas Temporais (5 minutos)

## 1. Contexto

Após a limpeza e normalização temporal dos dados (NB02), este notebook realiza a discretização da linha do tempo em janelas fixas, permitindo a análise agregada do comportamento do sistema.

A modelagem baseada em eventos individuais apresenta limitações para análise temporal contínua. Assim, torna-se necessário transformar os dados em uma série temporal estruturada.



## 2. Objetivo

O objetivo deste notebook é:

- Discretizar o tempo em janelas fixas de 5 minutos;
- Agregar eventos dentro de cada janela;
- Construir métricas representativas do comportamento do sistema;
- Produzir uma série temporal contínua adequada para análise estatística e modelagem.



## 3. Papel no Pipeline Atualizado

Este notebook representa a transição do nível de eventos individuais para o nível de comportamento agregado do sistema.

No contexto da modelagem de transições entre estados operacionais, esta etapa é fundamental, pois:

- estabelece a unidade temporal de análise;
- viabiliza a definição de estados por janela;
- permite a derivação de relações temporais entre janelas consecutivas.

A partir desta estrutura, torna-se possível modelar relações do tipo:

\[
state_t \rightarrow state_{t+1}
\]

que serão formalizadas posteriormente no NB06.



## 4. Principais Transformações

As transformações realizadas incluem:

- Definição de janelas fixas de 5 minutos;
- Mapeamento de eventos para identificadores de janela (`bucket_id`);
- Agregação de métricas por janela;
- Construção de uma série temporal contínua, incluindo janelas sem eventos.



## 5. Estrutura das Métricas

As métricas agregadas por janela incluem:

### Volume
- Número total de eventos;
- Número de falhas;
- Número de máquinas;
- Número de coleções.

### Intensidade
- Média de prioridade;
- Média de CPU solicitada;
- Média de memória solicitada.

### Tipologia de eventos
- Contagem de eventos por tipo (FAIL, SCHEDULE, FINISH, etc.).



## 6. Saídas Esperadas

Ao final deste notebook, espera-se obter:

- Dataset agregado por janelas de 5 minutos;
- Série temporal contínua (incluindo janelas vazias);
- Base estruturada para detecção de episódios (NB04);
- Dados preparados para engenharia de atributos (NB05).



## 7. Relação com as Próximas Etapas

Este notebook alimenta diretamente:

- NB04 — Detecção de episódios críticos;
- NB05 — Engenharia de atributos;
- NB06 — Derivação de estados e transições.

A qualidade da discretização e agregação impacta diretamente todas as etapas subsequentes.



## 8. Observações Metodológicas

- A escolha da janela de 5 minutos representa um compromisso entre granularidade e estabilidade;
- A inclusão de janelas sem eventos garante continuidade temporal;
- A agregação reduz ruído e permite análise estatística mais robusta.

Este notebook estabelece a base formal da série temporal utilizada no restante do pipeline.

In [ ]:
# ============================================================
# 03_window_5min_base.ipynb
# ============================================================

# =========================
# Bootstrap Seguro
# =========================
from pathlib import Path
import sys, subprocess, importlib

# Mount seguro (não quebra se já estiver montado)
if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("[Bootstrap] Drive já montado.")

REPO_DIR = Path("/content/drive/MyDrive/Mestrado/PPCOMP_DM")
GITHUB_REPO = "https://github.com/sergiocostaifes/PPCOMP_DM.git"

if not REPO_DIR.exists():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", GITHUB_REPO, str(REPO_DIR)], check=True)

repo_str = str(REPO_DIR)
if repo_str not in sys.path:
    sys.path.insert(0, repo_str)

importlib.invalidate_caches()

from src.paths import PROCESSED_PATH, FEATURES_PATH, REPORTS_PATH, ensure_dirs
ensure_dirs()

def log(msg):
    print(f"[03_window_5min_base] {msg}")


# =========================
# Parâmetros
# =========================
BUCKET_SEC = 5 * 60
BUCKET_US = BUCKET_SEC * 1_000_000

TOP_EVENTS = ["FAIL", "SCHEDULE", "FINISH", "ENABLE", "LOST", "EVICT", "KILL"]


# =========================
# 1) Leitura
# =========================
import pandas as pd
import numpy as np
import json

CLEAN_PARQUET = PROCESSED_PATH / "google_trace_clean.parquet"
assert CLEAN_PARQUET.exists(), f"Arquivo não encontrado: {CLEAN_PARQUET}"

df = pd.read_parquet(CLEAN_PARQUET)

log(f"Shape entrada: {df.shape}")

required_cols = ["t_rel_us", "machine_id", "collection_id", "event", "failed"]
missing = [c for c in required_cols if c not in df.columns]
assert not missing, f"Colunas ausentes: {missing}"

df["t_rel_us"] = pd.to_numeric(df["t_rel_us"], errors="coerce")
df = df.dropna(subset=["t_rel_us"]).copy()
df["t_rel_us"] = df["t_rel_us"].astype("int64")


# =========================
# 2) Minute Bucket
# =========================
df["bucket_id"] = (df["t_rel_us"] // BUCKET_US).astype("int64")
df["bucket_start_us"] = df["bucket_id"] * BUCKET_US

log(f"bucket range: {df['bucket_id'].min()}..{df['bucket_id'].max()}")
log(f"Buckets distintos: {df['bucket_id'].nunique()}")


# =========================
# 3) resource_request (leve + diagnóstico)
# =========================
import ast

def parse_dict(x):
    """
    Converte resource_request para dict quando possível.
    Suporta:
      - dict já pronto
      - string JSON (aspas duplas)
      - string estilo Python dict (aspas simples) via ast.literal_eval
    Retorna None se não conseguir parsear ou se não for dict.
    """
    if isinstance(x, dict):
        return x

    if isinstance(x, str):
        s = x.strip()
        if not s:
            return None

        # 1) JSON padrão
        try:
            v = json.loads(s)
            return v if isinstance(v, dict) else None
        except Exception:
            pass

        # 2) Fallback para "{'cpus': 1, 'memory': 2}"
        try:
            v = ast.literal_eval(s)
            return v if isinstance(v, dict) else None
        except Exception:
            return None

    return None

# Extrai cpus/memory quando existirem
if "resource_request" in df.columns:
    rr = df["resource_request"].map(parse_dict)
    df["req_cpus"] = rr.map(lambda d: d.get("cpus") if isinstance(d, dict) else np.nan)
    df["req_mem"]  = rr.map(lambda d: d.get("memory") if isinstance(d, dict) else np.nan)
else:
    df["req_cpus"] = np.nan
    df["req_mem"]  = np.nan

# Diagnóstico de qualidade (não mascarar NaN!)
valid_cpu = int(df["req_cpus"].notna().sum())
valid_mem = int(df["req_mem"].notna().sum())

log(f"req_cpus válidos: {valid_cpu} ({valid_cpu/len(df):.4%})")
log(f"req_mem  válidos: {valid_mem} ({valid_mem/len(df):.4%})")

# Indicador binário (útil para agregação e para decidir se a feature vale a pena)
df["has_req_cpus"] = df["req_cpus"].notna().astype("int64")
df["has_req_mem"]  = df["req_mem"].notna().astype("int64")


# =========================
# 4) Agregações
# =========================
base = (
    df.groupby("bucket_id", as_index=False)
      .agg(
          bucket_start_us=("bucket_start_us", "min"),
          n_events=("event", "size"),
          n_failed=("failed", "sum"),
          n_machines=("machine_id", "nunique"),
          n_collections=("collection_id", "nunique"),
          mean_priority=("priority", "mean"),
          mean_req_cpus=("req_cpus", "mean"),
          mean_req_mem=("req_mem", "mean"),
# taxa de presença por bucket (não confundir com média!)
          req_cpus_presence_rate=("has_req_cpus", "mean"),
          req_mem_presence_rate=("has_req_mem", "mean"),
      )
)

evt_counts = (
    df[df["event"].isin(TOP_EVENTS)]
      .groupby(["bucket_id", "event"], observed=False)
      .size()
      .unstack(fill_value=0)
)

for ev in TOP_EVENTS:
    if ev not in evt_counts.columns:
        evt_counts[ev] = 0

evt_counts = evt_counts[TOP_EVENTS].reset_index()
evt_counts = evt_counts.rename(columns={ev: f"event_{ev}_count" for ev in TOP_EVENTS})

base = base.merge(evt_counts, on="bucket_id", how="left")

count_cols = ["n_events", "n_failed", "n_machines", "n_collections"] + \
             [f"event_{ev}_count" for ev in TOP_EVENTS]

for c in count_cols:
    base[c] = base[c].fillna(0).astype("int64")

base = base.sort_values("bucket_id").reset_index(drop=True)

log(f"Base shape: {base.shape}")


# =========================
# 5) Série contínua
# =========================
bmin = int(base["bucket_id"].min())
bmax = int(base["bucket_id"].max())

full = pd.DataFrame({"bucket_id": np.arange(bmin, bmax + 1)})
full["bucket_start_us"] = full["bucket_id"] * BUCKET_US

series = full.merge(base, on=["bucket_id", "bucket_start_us"], how="left")

for c in count_cols:
    series[c] = series[c].fillna(0).astype("int64")

log(f"Série shape: {series.shape}")


# =========================
# 6) Persistência
# =========================
BASE_FILE = FEATURES_PATH / "window_5min_base.parquet"
SERIES_FILE = FEATURES_PATH / "window_5min_series.parquet"

base.to_parquet(BASE_FILE, compression="snappy", index=False)
series.to_parquet(SERIES_FILE, compression="snappy", index=False)

summary = {
    "rows_in": int(len(df)),
    "base_rows": int(len(base)),
    "series_rows": int(len(series)),
    "bucket_id_min": bmin,
    "bucket_id_max": bmax,
    "bucket_span": int(bmax - bmin + 1),
    "gap_buckets": int(len(series) - len(base)),
    "avg_events_per_bucket": float(base["n_events"].mean()),
    "avg_failed_per_bucket": float(base["n_failed"].mean()),
    # Qualidade das features de resource_request
    "req_cpu_valid_ratio": float(valid_cpu / len(df)),
    "req_mem_valid_ratio": float(valid_mem / len(df)),
    "top_events": TOP_EVENTS,
}

summary_file = REPORTS_PATH / "03_window_5min_base_summary.json"
summary_file.write_text(json.dumps(summary, indent=2, ensure_ascii=False))

log("Notebook 03 finalizado com sucesso.")
base.head()

Mounted at /content/drive
[03_window_5min_base] Shape entrada: (349151, 23)
[03_window_5min_base] bucket range: 12..8929
[03_window_5min_base] Buckets distintos: 8917
[03_window_5min_base] req_cpus válidos: 349151 (100.0000%)
[03_window_5min_base] req_mem  válidos: 349151 (100.0000%)
[03_window_5min_base] Base shape: (8917, 18)
[03_window_5min_base] Série shape: (8918, 18)
[03_window_5min_base] Notebook 03 finalizado com sucesso.


,bucket_id,bucket_start_us,n_events,n_failed,n_machines,n_collections,mean_priority,mean_req_cpus,mean_req_mem,req_cpus_presence_rate,req_mem_presence_rate,event_FAIL_count,event_SCHEDULE_count,event_FINISH_count,event_ENABLE_count,event_LOST_count,event_EVICT_count,event_KILL_count
0,12,3600000000,25,6,25,20,270.160000,0.019362,0.020901,1.0,1.0,6,2,2,4,11,0,0
1,13,3900000000,46,13,43,23,209.717391,0.007103,0.018461,1.0,1.0,13,0,13,7,12,0,1
2,14,4200000000,34,11,32,20,189.382353,0.014036,0.015593,1.0,1.0,11,1,10,7,5,0,0
3,15,4500000000,44,10,41,29,249.522727,0.008850,0.003220,1.0,1.0,10,0,9,10,15,0,0
4,16,4800000000,22,5,22,19,219.045455,0.011749,0.003593,1.0,1.0,5,1,6,4,5,0,1


## 9. Conclusão da Etapa

A etapa de construção de janelas temporais foi concluída com sucesso, resultando em uma representação agregada e contínua do comportamento do sistema ao longo do tempo.

A transformação de eventos individuais em uma série temporal estruturada permite:

- análise estatística consistente;
- detecção de padrões temporais;
- preparação para modelagem supervisionada.



## 10. Importância para a Modelagem de Transições

A série temporal construída neste notebook constitui a base para a definição de estados operacionais e para a modelagem de transições entre estados.

A estrutura contínua de janelas permite estabelecer relações entre instantes consecutivos, fundamentais para análises do tipo:

- BEFORE → DURING  
- BEFORE → NORMAL  

Sem essa discretização e continuidade, a modelagem de transições não seria viável de forma consistente.



## 11. Encaminhamento do Pipeline

Com a série temporal construída, o pipeline segue para:

➡️ NB04 — Detecção de episódios críticos  
➡️ NB05 — Engenharia de atributos  
➡️ NB06 — Derivação de estados e transições  



## 12. Achados Experimentais (Execução Atual)

Os resultados desta execução indicam:

### Entrada
- Arquivo de entrada: `google_trace_clean.parquet`
- Registros processados: **349.151**



### Discretização Temporal
- Janela fixa adotada: **5 minutos (300 segundos)**
- Conversão: `bucket_id = floor(t_rel_us / 300000000)`
- Intervalo de buckets observado: **12..8929**
- Total de buckets com eventos: **8.917**



### Estrutura da Série Temporal
- Intervalo total de buckets considerado: **8.918**
- Janelas sem eventos foram preenchidas com zero nas métricas de contagem
- Série contínua pronta para:
  - janelas móveis (rolling windows)
  - detecção de episódios
  - modelagem supervisionada



### Métricas Agregadas

#### Volume
- `n_events`
- `n_failed`
- `n_machines`
- `n_collections`

#### Intensidade
- `mean_priority`
- `mean_req_cpus`
- `mean_req_mem`

#### Tipologia de eventos
- Contagens por tipo de evento (FAIL, SCHEDULE, FINISH, ENABLE, LOST, EVICT, KILL)



### Consistência de Recursos
- `req_cpus` válidos: **100%**
- `req_mem` válidos: **100%**
- Presença registrada por:
  - `req_cpus_presence_rate`
  - `req_mem_presence_rate`



### Persistência
- `window_5min_base.parquet`
- `window_5min_series.parquet`
- `03_window_5min_base_summary.json`



### Conclusão Experimental

Esta etapa consolida a transição:

- do nível de eventos individuais;
- para o nível de comportamento agregado do sistema.

A série temporal contínua resultante constitui a base formal para:

- detecção de episódios (NB04);
- engenharia de atributos (NB05);
- modelagem preditiva e de transições (NB06).